In [ ]:
import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit, ols

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# загружен датасеты в виде словаря
path_to_datasets = '../datasets/'
df_dict = {'loanapp': pd.read_csv(path_to_datasets+'loanapp.csv'), 
		'TableF5-1':pd.read_csv(path_to_datasets+'TableF5-1.csv'), 
		'SwissLabour': pd.read_csv(path_to_datasets+'SwissLabour.csv')}

In [ ]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['I(appinc/100)+mortno+unem+dep+male',
			'WA+np.log(FAMINC)+WE+KL6+K618+CIT+UN',
			'income+I(income**2)+age+I(age**2)+youngkids+oldkids']

mod_name = ['Approve equation', 'labour force equation', 'swiss labour force equation']

new_data = [ pd.DataFrame( {'appinc':[120, 48, 82], 'mortno':[1, 1, 0], 'unem':[1.8, 3.2, 3.9], 'dep':[0,0,1], 'married':[1, 0, 1], 'male':[1, 0, 1]} ), 
			pd.DataFrame( {'WA':[34, 36, 42], 'CIT':[1, 0, 1], 'UN':[5, 7.5, 3], 'FAMINC':[12500, 9800, 67800], 'KL6':[2, 1, 2], 'K618':[0, 2, 1], 'WE':[15, 12, 14] } ),
			pd.DataFrame( {'income':[1.367, 9.217, 10.686], 'age':[2.5, 3.7, 4.2], 'youngkids':[0, 2, 2], 'oldkids':[0,0,1]} ) ]


digits = 3

for i, df_name in enumerate(df_names):
	mod_logit  = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_probit = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_ols = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res_logit = mod_logit.fit(disp=False)
	res_probit = mod_probit.fit(disp=False)
	res_ols = mod_ols.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'Для датасет `{df_name}` оцените __probit__  и __logit__ регрессии `{endog_var[i]}` на `{exog_vars[i]}` следующих спецификаций:\n'
	f'- LPM \n'
	f'- logit\n '
	f'- probit\n\n '
	f'Рассмотрим новые наблюдения:'))
	display(Markdown(new_data[i].to_markdown(index=False)))
	display(Markdown(f'''Постройте прогноз для каждого наблюдения и дайте интерпретацию. __Ответ округлите до {digits}-х десятичных знаков__ '''))

	y_pred_df = pd.DataFrame({'LPM':res_ols.predict(exog=new_data[i], transform=True).round(digits), 
								'logit':res_logit.predict(exog=new_data[i], transform=True).round(digits),
								'probit':res_probit.predict(exog=new_data[i], transform=True).round(digits) } )

	display(Markdown(f'''Ответ:\n'''))
	display(Markdown(y_pred_df.to_markdown(index=False)))

